# Lab 03 - Feature Engineering Pipeline with TFX

A port of the TFX feature engineering lab to the **Wisconsin Diagnostic Breast
Cancer** dataset. The original runs UCI Census Income through five components;
this one runs the same five over data that is entirely numeric, which changes
what `preprocessing_fn` has to do.

You will build only the feature engineering half of a pipeline:

1. **`CsvExampleGen`** - ingest the CSV and split it into train and eval
2. **`StatisticsGen`** - compute descriptive statistics over each split
3. **`SchemaGen`** - infer types, domains and value ranges from those statistics
4. **`ExampleValidator`** - check the eval split against the schema for anomalies
5. **`Transform`** - apply `preprocessing_fn` to produce model-ready features

There is deliberately no `Trainer`, `Evaluator` or `Pusher`. The lab stops where
features are materialised.

## Setup

### Import packages

In [1]:
import os
import pprint

import tensorflow as tf
from tfx import v1 as tfx
from tfx.orchestration.experimental.interactive.interactive_context import (
    InteractiveContext,
)

pp = pprint.PrettyPrinter(indent=2)

print("TensorFlow version:", tf.__version__)
print("TFX version:", tfx.__version__)

2026-11-02 00:42:33.651320: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-02 00:42:33.655946: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-02 00:42:33.663123: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-02 00:42:33.680202: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-02 00:42:33.713689: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

/usr/local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.10.22). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)


2026-11-02 00:42:39.846876: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


/usr/local/lib/python3.10/site-packages/google/auth/transport/grpc.py:43: FutureWarning: grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, google-auth will raise its minimum requirements to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(


/usr/local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.10.22). Google will not post any further updates to google.cloud.bigquery_storage_v1 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.bigquery_storage_v1.
  warnings.warn(message, FutureWarning)


/usr/local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.10.22). Google will not post any further updates to google.cloud.bigquery supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.bigquery.
  warnings.warn(message, FutureWarning)


... (5 outputs left out)

/usr/local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.10.22). Google will not post any further updates to google.cloud.aiplatform.v1.schema.predict.prediction_v1 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.aiplatform.v1.schema.predict.prediction_v1.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.10.22). Google will not post any further updates to google.cloud.aiplatform.v1.schema.trainingjob.definition_v1 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.aiplatform.v1.schema.trainingjob.definition_v1.
  warnings.warn(message, FutureWarning)


/usr/local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.10.22). Google will not post any further updates to google.cloud.aiplatform.v1beta1.schema.predict.instance_v1beta1 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.aiplatform.v1beta1.schema.predict.instance_v1beta1.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.10.22). Google will not post any further updates to google.cloud.aiplatform.v1beta1.schema.predict.params_v1beta1 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.aiplatform.v1beta1.schema.predict.params_v1beta1.
  warnings.warn(message, FutureWarning)


/usr/local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.10.22). Google will not post any further updates to google.cloud.aiplatform.v1beta1.schema.predict.prediction_v1beta1 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.aiplatform.v1beta1.schema.predict.prediction_v1beta1.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.10.22). Google will not post any further updates to google.cloud.aiplatform.v1beta1.schema.trainingjob.definition_v1beta1 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.aiplatform.v1beta1.schema.trainingjob.definition_v1beta1.
  warnings.warn(message, FutureWarning)


TensorFlow version: 2.16.2
TFX version: 1.16.0


### Define paths

`CsvExampleGen` takes an `input_base` **directory**, not a file path - it reads
every CSV it finds there. That is why the cleaned data lives in a subdirectory
of its own rather than sitting loose in `data/`.

In [2]:
# Directory CsvExampleGen scans. Must contain only the data CSV.
_data_root = "./data/breast_cancer"
_data_filepath = os.path.join(_data_root, "breast_cancer_data.csv")

# Where InteractiveContext writes every component's output.
_pipeline_root = "./pipeline/"

# Passed to Transform as module_file. A real path, not an import.
_transform_module_file = "breast_cancer_transform.py"

assert os.path.exists(_data_filepath), (
    f"{_data_filepath} is missing. Run `python prepare_data.py` first."
)
print("data root:", _data_root)

data root: ./data/breast_cancer


### Preview the dataset

569 samples, 32 columns: an `id`, a `diagnosis` label (`M`/`B`), and 30 numeric
features. Those 30 are 10 nucleus measurements - radius, texture, perimeter,
area, smoothness, compactness, concavity, concave points, symmetry, fractal
dimension - each recorded three ways: `_mean`, `_se` (standard error) and
`_worst` (mean of the three largest values).

In [3]:
import pandas as pd

preview = pd.read_csv(_data_filepath)
print("shape:", preview.shape)
print()
print("label balance:")
print(preview["diagnosis"].value_counts())
preview.head()

shape: (569, 32)

label balance:
B    357
M    212
Name: diagnosis, dtype: int64


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


Note the spread. `area_worst` and `fractal_dimension_se` differ by about four
orders of magnitude - which is what makes the scaling choice in
`preprocessing_fn` consequential rather than cosmetic.

In [4]:
spread = preview.drop(columns=["id", "diagnosis"]).agg(["min", "max"]).T
spread["range"] = spread["max"] - spread["min"]
spread.sort_values("range", ascending=False).head(8)

,min,max,range
area_worst,185.200,4254.00,4068.800
area_mean,143.500,2501.00,2357.500
area_se,6.802,542.20,535.398
perimeter_worst,50.410,251.20,200.790
perimeter_mean,43.790,188.50,144.710
texture_worst,12.020,49.54,37.520
texture_mean,9.710,39.28,29.570
radius_worst,7.930,36.04,28.110


### Create the InteractiveContext

`InteractiveContext` is the notebook-mode orchestrator. It runs one component at
a time and records each output in a local ML Metadata store, so later components
can look up what earlier ones produced. A production pipeline swaps this for
Airflow, Beam or Kubeflow - the component code is unchanged.

Leaving the metadata path unset puts the store under `_pipeline_root`.

In [5]:
context = InteractiveContext(pipeline_root=_pipeline_root)

## Run TFX components interactively

### ExampleGen

`CsvExampleGen` reads the CSV, converts each row to a `tf.train.Example`, and
splits the result. The default is a 2:1 train/eval hash split - note this is a
*hash* of each record, not a shuffle, so the split is deterministic across runs.

It also infers each column's type from the CSV header and values. This is the
first place `id` enters the pipeline as if it were a feature.

In [6]:
example_gen = tfx.components.CsvExampleGen(input_base=_data_root)
context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 1
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [7]:
artifact = example_gen.outputs["examples"].get()[0]
print("split names:", artifact.split_names)
print("uri:", artifact.uri)

split names: ["train", "eval"]
uri: ./pipeline/CsvExampleGen/examples/1


Read a record back to confirm the split is populated:

In [8]:
train_uri = os.path.join(artifact.uri, "Split-train")
tfrecord_files = [os.path.join(train_uri, name) for name in os.listdir(train_uri)]
dataset = tf.data.TFRecordDataset(tfrecord_files, compression_type="GZIP")

for raw_record in dataset.take(1):
    example = tf.train.Example()
    example.ParseFromString(raw_record.numpy())
    for key in sorted(example.features.feature)[:6]:
        print(key, "->", example.features.feature[key])

area_mean -> float_list {
  value: 1326.0
}

area_se -> float_list {
  value: 74.08000183105469
}

area_worst -> float_list {
  value: 1956.0
}

compactness_mean -> float_list {
  value: 0.07863999903202057
}

compactness_se -> float_list {
  value: 0.013079999946057796
}

compactness_worst -> float_list {
  value: 0.186599999666214
}



2026-11-02 00:43:24.492754: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### StatisticsGen

Computes descriptive statistics per split - counts, missing values, means,
quantiles, histograms. This is TensorFlow Data Validation (TFDV) running under
the hood. Everything downstream depends on these: `SchemaGen` infers a schema
from them, and `ExampleValidator` compares splits against that schema.

In [9]:
statistics_gen = tfx.components.StatisticsGen(
    examples=example_gen.outputs["examples"]
)
context.run(statistics_gen)

ExecutionResult(
    component_id: StatisticsGen
    execution_id: 2
    outputs:
        statistics: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [10]:
context.show(statistics_gen.outputs["statistics"])

In the rendered view, switch between the train and eval splits. Two things worth
checking: that `diagnosis` has exactly two values, and that `id` is being treated
as an ordinary numeric feature - because nothing so far knows it is not one.

### SchemaGen

Infers a schema from the statistics: each feature's type, whether it is
required, and its expected domain or value range. The schema becomes the
contract that incoming data is checked against.

In [11]:
schema_gen = tfx.components.SchemaGen(
    statistics=statistics_gen.outputs["statistics"],
    infer_feature_shape=True,
)
context.run(schema_gen)

ExecutionResult(
    component_id: SchemaGen
    execution_id: 3
    outputs:
        schema: OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [12]:
context.show(schema_gen.outputs["schema"])

,Type,Presence,Valency,Domain
Feature name,,,,
'area_mean',FLOAT,required,,-
'area_se',FLOAT,required,,-
'area_worst',FLOAT,required,,-
'compactness_mean',FLOAT,required,,-
'compactness_se',FLOAT,required,,-
'compactness_worst',FLOAT,required,,-
'concave points_mean',FLOAT,required,,-
'concave points_se',FLOAT,required,,-
'concave points_worst',FLOAT,required,,-


,Values
Domain,
'diagnosis',"'B', 'M'"


`diagnosis` should appear as a `BYTES` feature with domain `{'B', 'M'}`, and the
30 measurements as `FLOAT`. `id` appears as `INT` with a very wide range.

An inferred schema is a *starting point*, not an authority - it describes
whatever it was shown. In a real project you would curate it by hand and check
it into version control.

### ExampleValidator

Compares the statistics against the schema and reports anomalies: features that
are missing, out of their declared domain, or wrongly typed. In production this
is the gate that catches an upstream change before it reaches training.

In [13]:
example_validator = tfx.components.ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"],
)
context.run(example_validator)

ExecutionResult(
    component_id: ExampleValidator
    execution_id: 4
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [14]:
context.show(example_validator.outputs["anomalies"])

Expect "No anomalies found" - the schema was inferred from this same data, so it
would be surprising to find any. The check earns its keep on the *next* batch,
not this one.

To see it actually fire, narrow a feature's range in the schema and re-run the
validator.

### Transform

The component this lab is really about. `Transform` runs `preprocessing_fn` once
to build a TensorFlow graph, lets the `tft.*` analyzers compute their statistics
over the **training** split, bakes those in as graph constants, and then replays
that graph over every split.

That is the whole point: the eval split is scaled with *training* statistics,
never its own. The same graph ships with the model, so serving-time preprocessing
is identical to training-time by construction rather than by discipline.

In [15]:
print(open(_transform_module_file).read())

"""preprocessing_fn for the breast cancer feature engineering pipeline.

Transform runs this once over the full training split to compute whatever
statistics the tft.* analyzers need (means, variances, min/max, quantile
boundaries, vocabularies), bakes those constants into a TensorFlow graph, and
then replays that graph over every split. That is what makes the training and
serving paths agree: the eval split is scaled with the *training* statistics,
never its own.
"""

import tensorflow as tf
import tensorflow_transform as tft

import breast_cancer_constants

_MEAN_FEATURE_KEYS = breast_cancer_constants.MEAN_FEATURE_KEYS
_SE_FEATURE_KEYS = breast_cancer_constants.SE_FEATURE_KEYS
_WORST_FEATURE_KEYS = breast_cancer_constants.WORST_FEATURE_KEYS
_BUCKET_FEATURE_KEYS = breast_cancer_constants.BUCKET_FEATURE_KEYS
_FEATURE_BUCKET_COUNT = breast_cancer_constants.FEATURE_BUCKET_COUNT
_LABEL_KEY = breast_cancer_constants.LABEL_KEY
_transformed_name = breast_cancer_constants.transformed_name
_bu

In [16]:
transform = tfx.components.Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=os.path.abspath(_transform_module_file),
)
context.run(transform)

running bdist_wheel
running build
running build_py
creating build/lib
copying breast_cancer_constants.py -> build/lib
copying breast_cancer_transform.py -> build/lib
copying prepare_data.py -> build/lib
installing to /tmp/tmpxk4c2et8
running install
running install_lib



/usr/local/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        This deprecation is overdue, please update your project and remove deprecated
        calls to avoid build errors in the future.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()



copying build/lib/breast_cancer_constants.py -> /tmp/tmpxk4c2et8/.
copying build/lib/prepare_data.py -> /tmp/tmpxk4c2et8/.
copying build/lib/breast_cancer_transform.py -> /tmp/tmpxk4c2et8/.
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'


reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/tmpxk4c2et8/./tfx_user_code_Transform-0.0+8bce1c1e41a5c8a1f0bf743e696fafec89c2f4360826306e84e217cd6f6a3560-py3.10.egg-info
running install_scripts
creating /tmp/tmpxk4c2et8/tfx_user_code_transform-0.0+8bce1c1e41a5c8a1f0bf743e696fafec89c2f4360826306e84e217cd6f6a3560.dist-info/WHEEL
creating '/tmp/tmp3xzx61tr/tfx_user_code_transform-0.0+8bce1c1e41a5c8a1f0bf743e696fafec89c2f4360826306e84e217cd6f6a3560-py3-none-any.whl' and adding '/tmp/tmpxk4c2et8' to it
adding 'breast_cancer_constants.py'
adding 'breast_cancer_transform.py'
adding 'prepare_data.py'
adding 'tfx_user_code_transform-0.0+8bce1c1e41a5c8a1f0bf743e696fafec89c2f4360826306e84e217cd6f6a3560.dist-info/METADATA'
adding 'tfx_user_code_transform-0.0+8bce1c1e41a5c8a1f0bf743e696fafec89c2f4360826306e84e217cd6f6a3560.dist-info/WHEEL'
adding 'tfx_user_code

Processing ./pipeline/_wheels/tfx_user_code_transform-0.0+8bce1c1e41a5c8a1f0bf743e696fafec89c2f4360826306e84e217cd6f6a3560-py3-none-any.whl


Processing ./pipeline/_wheels/tfx_user_code_transform-0.0+8bce1c1e41a5c8a1f0bf743e696fafec89c2f4360826306e84e217cd6f6a3560-py3-none-any.whl


... (20 outputs left out)

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 5
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

Three outputs: `transform_graph` (the reusable graph, what ships with a model),
`transformed_examples` (the materialised features) and `updated_analyzer_cache`.

In [17]:
pp.pprint(transform.outputs)

{ 'post_transform_anomalies': OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False),
  'post_transform_schema': OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False),
  'post_transform_stats': OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False),
  'pre_transform_schema': OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False),
  'pre_transform_stats': OutputChannel(artifact_type=ExampleStati

Read back a transformed record to confirm the feature names:

In [18]:
transform_uri = transform.outputs["transformed_examples"].get()[0].uri
transformed_train = os.path.join(transform_uri, "Split-train")
files = [os.path.join(transformed_train, n) for n in os.listdir(transformed_train)]

dataset = tf.data.TFRecordDataset(files, compression_type="GZIP")
for raw_record in dataset.take(1):
    example = tf.train.Example()
    example.ParseFromString(raw_record.numpy())
    keys = sorted(example.features.feature)
    print("transformed feature count:", len(keys))
    print()
    pp.pprint(keys)

transformed feature count: 32

[ 'area_mean_xf',
  'area_se_xf',
  'area_worst_xf',
  'compactness_mean_xf',
  'compactness_se_xf',
  'compactness_worst_xf',
  'concave points_mean_xf',
  'concave points_se_xf',
  'concave points_worst_xf',
  'concavity_mean_xf',
  'concavity_se_xf',
  'concavity_worst_xf',
  'diagnosis_xf',
  'fractal_dimension_mean_xf',
  'fractal_dimension_se_xf',
  'fractal_dimension_worst_xf',
  'perimeter_mean_xf',
  'perimeter_se_xf',
  'perimeter_worst_xf',
  'radius_mean_bucketized_xf',
  'radius_mean_xf',
  'radius_se_xf',
  'radius_worst_xf',
  'smoothness_mean_xf',
  'smoothness_se_xf',
  'smoothness_worst_xf',
  'symmetry_mean_xf',
  'symmetry_se_xf',
  'symmetry_worst_xf',
  'texture_mean_xf',
  'texture_se_xf',
  'texture_worst_xf']


2026-11-02 00:46:20.427540: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Check three things:

- every name ends in `_xf`, so nothing untransformed leaked through
- **`id` is absent** - `SchemaGen` described it as a feature, and
  `preprocessing_fn` dropped it on purpose
- `radius_mean` appears twice: once scaled (`radius_mean_xf`) and once
  bucketized (`radius_mean_bucketized_xf`)

In [19]:
import statistics as stats

values = {}
for raw_record in tf.data.TFRecordDataset(files, compression_type="GZIP").take(200):
    example = tf.train.Example()
    example.ParseFromString(raw_record.numpy())
    for key, feature in example.features.feature.items():
        if feature.float_list.value:
            values.setdefault(key, []).append(feature.float_list.value[0])

for key in sorted(values)[:10]:
    column = values[key]
    print(
        f"{key:34} min={min(column):8.3f} "
        f"max={max(column):8.3f} mean={stats.mean(column):8.3f}"
    )

area_mean_xf                       min=   0.000 max=   0.999 mean=   0.223
area_se_xf                         min=  -0.648 max=   9.441 mean=   0.041
area_worst_xf                      min=  -1.198 max=   4.376 mean=   0.045
compactness_mean_xf                min=   0.000 max=   1.000 mean=   0.309
compactness_se_xf                  min=  -1.270 max=   5.962 mean=   0.084
compactness_worst_xf               min=  -1.313 max=   4.805 mean=   0.080
concave points_mean_xf             min=   0.000 max=   1.000 mean=   0.264
concave points_se_xf               min=  -1.952 max=   4.885 mean=   0.031
concave points_worst_xf            min=  -1.677 max=   2.586 mean=   0.062
concavity_mean_xf                  min=   0.000 max=   1.000 mean=   0.224


2026-11-02 00:46:20.652398: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


The ranges here are the direct consequence of the scaling choice, and the two
families should look visibly different:

- the `_mean` features were scaled with `tft.scale_to_0_1`, so they sit in
  [0, 1]
- the `_se` and `_worst` features were scaled with `tft.scale_to_z_score`, so
  they centre near 0 and run negative

That split is deliberate. `_mean` is the best-behaved of the three families, so
a bounded range costs nothing. `_se` is right-skewed and `_worst` is by
construction the extreme-value tail of each measurement -- for both, min-max
would let one large value pull the range out and compress everything else into
a narrow band.

Worth re-running with all 30 scaled the same way to see what that costs.

---

## Where this stops

Features are materialised and the transform graph is saved. A full pipeline
would continue with `Trainer`, `Evaluator` and `Pusher` - none of which are
here, by design.

**Optional:** narrow the schema so `ExampleValidator` actually reports an
anomaly, or swap the scaling strategy and compare the printed ranges.